In [1]:
!pip install mediapipe==0.10.14 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 53.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 14.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
a2a-sdk 0.3.26 requires protobuf>=5.29.5, but you have protobuf 4.25.9 which is incompatible.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have proto

In [2]:
!pip install protobuf==5.28.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 5.6 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
mediapipe 0.10.14 requires protobuf<5,>=4.25.3, but you have protobuf 5.28.3 which is incompatible.
a2a-sdk 0.3.26 requires protobuf>=5.29.5, but you have protobuf 5.28.3 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 5.28.3 which is incompatible.


In [5]:
import cv2
import mediapipe as mp
import numpy as np

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

In [6]:
print(mp.__version__)

0.10.14


In [7]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/aaminajadoon/gym-video/single_men.mp4


In [8]:
def calculate_angle(a, b, c):
    # a, b, c are each an x and y coordinate
    # b is the joint we are measuring the angle at, like the elbow or the knee
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    # build the two vectors going out from the middle point
    vector1 = a - b
    vector2 = c - b

    # cosine of the angle between two vectors is their dot product
    # divided by the product of their lengths
    cosine_angle = np.dot(vector1, vector2) / (np.linalg.norm(vector1) * np.linalg.norm(vector2))

    # numbers can sometimes slip just outside the arccos valid range due to rounding
    # so we clip it back into range before taking arccos
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)

    angle = np.degrees(np.arccos(cosine_angle))
    return angle

In [9]:
# each exercise lists which landmarks to use for the angle
# and the two thresholds that define the straight and bent positions
EXERCISES = {
    "bicep_curl": {
        "landmarks": (mp_pose.PoseLandmark.LEFT_SHOULDER,
                      mp_pose.PoseLandmark.LEFT_ELBOW,
                      mp_pose.PoseLandmark.LEFT_WRIST),
        "straight_angle": 160,
        "bent_angle": 40
    },
    "squat": {
        "landmarks": (mp_pose.PoseLandmark.LEFT_HIP,
                      mp_pose.PoseLandmark.LEFT_KNEE,
                      mp_pose.PoseLandmark.LEFT_ANKLE),
        "straight_angle": 165,
        "bent_angle": 100
    },
    "push_up": {
        "landmarks": (mp_pose.PoseLandmark.LEFT_SHOULDER,
                      mp_pose.PoseLandmark.LEFT_ELBOW,
                      mp_pose.PoseLandmark.LEFT_WRIST),
        "straight_angle": 160,
        "bent_angle": 90
    },
    # pull up uses the same elbow angle idea as bicep curl
    # arms straight when hanging at the bottom, arms bent when chin is near the bar at the top
    "pull_up": {
        "landmarks": (mp_pose.PoseLandmark.LEFT_SHOULDER,
                      mp_pose.PoseLandmark.LEFT_ELBOW,
                      mp_pose.PoseLandmark.LEFT_WRIST),
        "straight_angle": 160,
        "bent_angle": 70
    }
}

In [10]:
current_exercise = "pull_up"
target_reps = 3

counter = 0
stage = "up"

# this will store one entry per rep, with the video time it happened at
rep_log = []

video_path = "/kaggle/input/datasets/aaminajadoon/gym-video/single_men.mp4"
cap = cv2.VideoCapture(video_path)

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

output_path = "/kaggle/working/output_video.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            config = EXERCISES[current_exercise]
            p1, p2, p3 = config["landmarks"]

            point1 = [landmarks[p1.value].x, landmarks[p1.value].y]
            point2 = [landmarks[p2.value].x, landmarks[p2.value].y]
            point3 = [landmarks[p3.value].x, landmarks[p3.value].y]

            angle = calculate_angle(point1, point2, point3)

            if angle < config["bent_angle"]:
                stage = "down"
            if angle > config["straight_angle"] and stage == "down":
                stage = "up"
                counter += 1

                # cap.get with POS_MSEC gives us how far into the video we currently are, in milliseconds
                # we convert that into minutes and seconds so the log is easy to read
                current_time_seconds = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000
                minutes = int(current_time_seconds // 60)
                seconds = int(current_time_seconds % 60)
                timestamp_str = f"{minutes:02d}:{seconds:02d}"

                rep_log.append({
                    "rep_number": counter,
                    "video_timestamp": timestamp_str
                })

            mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

        cv2.putText(image, f"Exercise: {current_exercise}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(image, f"Reps: {counter}", (10, 70),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        if counter >= target_reps:
            cv2.putText(image, "Target reached, great work", (10, 110),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        out.write(image)

cap.release()
out.release()

print(f"Done. Total reps counted: {counter}")
print(f"Output saved to: {output_path}")

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1790502765.411118     195 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1790502765.449776     195 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Done. Total reps counted: 3
Output saved to: /kaggle/working/output_video.mp4


In [12]:
log_path = "/kaggle/working/rep_log.txt"

# this writes one readable line per rep into a plain text file
with open(log_path, "w") as f:
    f.write(f"Exercise: {current_exercise}\n")
    f.write(f"Target reps: {target_reps}\n")
    f.write(f"Total reps counted: {counter}\n")
    f.write("\n")
    for entry in rep_log:
        f.write(f"Rep {entry['rep_number']} completed at video time {entry['video_timestamp']}\n")

print(f"Log saved to: {log_path}")
print()

# this reads the file back and prints it, so you can see it directly in the notebook too
with open(log_path, "r") as f:
    print(f.read())

Log saved to: /kaggle/working/rep_log.txt

Exercise: pull_up
Target reps: 3
Total reps counted: 3

Rep 1 completed at video time 00:02
Rep 2 completed at video time 00:09
Rep 3 completed at video time 00:13



In [13]:
!ffmpeg -y -i /kaggle/working/output_video.mp4 -vcodec libx264 -pix_fmt yuv420p /kaggle/working/output_video_fixed.mp4

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [14]:
from IPython.display import Video

Video("/kaggle/working/output_video_fixed.mp4", embed=True, width=640)